# Name That LLM — Optimized character n-gram sweep

This notebook evaluates a lightweight authorship-attribution model under prompt-grouped cross-validation.

**Fixed features:** word TF-IDF + punctuation frequencies.
**Swept feature:** raw character TF-IDF n-gram range.
**Classifier:** class-balanced LinearSVC.

The word matrix, punctuation matrix, and grouped folds are reused across configurations to reduce redundant computation.


In [ ]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# ------------------------------
# Reproducible experiment config
# ------------------------------
DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
OUTPUT_PATH = Path('/kaggle/working/char_sweep_results.json')
N_SPLITS = 5
RANDOM_STATE = 42
WORD_NGRAM_RANGE = (1, 3)
CHAR_NGRAM_RANGES = [(1, 5), (2, 6), (3, 7), (2, 7)]
MIN_DF = 2
MAX_FEATURES = 120_000
SVC_C = 1.0
MAX_ITER = 1_500

assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found: {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
required = {'text', 'model', 'prompt_id'}
missing = required.difference(train.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'

texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

print(f'Train rows: {len(train):,}')
print(f'Classes: {len(np.unique(labels))}')
print(f'Prompt groups: {len(np.unique(groups))}')
print(f'Character ranges: {CHAR_NGRAM_RANGES}')


In [ ]:
# ------------------------------
# Feature engineering utilities
# ------------------------------
PUNCTUATION = tuple('.,;:!?()[]{}"\'/-')

def punctuation_matrix(items):
    """Return per-character punctuation frequencies, one row per response."""
    rows = []
    for text in items:
        denominator = max(len(text), 1)
        rows.append([text.count(char) / denominator for char in PUNCTUATION])
    return np.asarray(rows, dtype=np.float32)

def make_word_vectorizer():
    return TfidfVectorizer(
        analyzer='word',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def make_char_vectorizer(ngram_range):
    return TfidfVectorizer(
        analyzer='char',
        ngram_range=ngram_range,
        min_df=MIN_DF,
        sublinear_tf=True,
        max_features=MAX_FEATURES,
    )

def fit_score(X_train, y_train, X_valid, y_valid):
    model = LinearSVC(
        C=SVC_C,
        class_weight='balanced',
        max_iter=MAX_ITER,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_valid)
    return float(balanced_accuracy_score(y_valid, predictions))

punctuation = punctuation_matrix(texts)
folds = list(GroupKFold(n_splits=N_SPLITS).split(texts, labels, groups=groups))
print(f'Prepared {len(folds)} fixed prompt-grouped folds.')


In [ ]:
# ------------------------------
# Prompt-normalized response patterns
# ------------------------------
# Each prompt group contains responses from multiple models. We measure how a
# response differs from the other responses for the same prompt. This is a
# transductive, group-context experiment: at inference time it requires the
# complete prompt group to be available. No labels are used to build features.
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score

PATTERN_OUTPUT = Path('/kaggle/working/prompt_pattern_results.json')
FEATURE_NAMES = [
    'characters', 'words', 'sentences', 'punctuation', 'digits', 'uppercase',
    'question_marks', 'exclamation_marks', 'colons', 'semicolons',
    'brackets', 'markdown_markers', 'bullet_markers', 'newline_count',
]

def response_pattern(text):
    words = re.findall(r"\b[\w']+\b", text)
    return [
        float(len(text)),
        float(len(words)),
        float(len(re.findall(r'[.!?]+', text))),
        float(sum(text.count(c) for c in PUNCTUATION)),
        float(sum(c.isdigit() for c in text)),
        float(sum(c.isupper() for c in text)),
        float(text.count('?')),
        float(text.count('!')),
        float(text.count(':')),
        float(text.count(';')),
        float(sum(text.count(c) for c in '()[]{}')),
        float(sum(text.count(c) for c in '#*_`')),
        float(sum(text.count(c) for c in '*-•')),
        float(text.count('\\n')),
    ]

absolute = np.asarray([response_pattern(text) for text in texts], dtype=np.float32)
relative_rows = []
unique_groups = np.unique(groups)
for group_id in unique_groups:
    group_values = absolute[groups == group_id]
    group_mean = group_values.mean(axis=0)
    group_std = group_values.std(axis=0)
    group_std[group_std < 1e-6] = 1.0
    # Include both centered values and within-prompt z-scores.
    relative_rows.append((absolute[groups == group_id] - group_mean) / group_std)
relative = np.zeros_like(absolute)
for group_id, group_relative in zip(unique_groups, relative_rows):
    relative[groups == group_id] = group_relative

feature_sets = {
    'absolute': absolute,
    'prompt_relative': relative,
    'absolute_plus_relative': np.hstack([absolute, relative]),
}
folds = list(GroupKFold(n_splits=N_SPLITS).split(texts, labels, groups=groups))
results = {}
started = time.time()

for feature_name, features in feature_sets.items():
    scores = []
    for fold_number, (train_idx, valid_idx) in enumerate(folds, start=1):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(features[train_idx])
        X_valid = scaler.transform(features[valid_idx])
        model = LinearSVC(C=1.0, class_weight='balanced', max_iter=2_000)
        model.fit(X_train, labels[train_idx])
        score = float(balanced_accuracy_score(labels[valid_idx], model.predict(X_valid)))
        scores.append(score)
        print(f'{feature_name} | fold {fold_number}/{N_SPLITS} | score={score:.4f}')
    results[feature_name] = {
        'mean_balanced_accuracy': float(np.mean(scores)),
        'std_balanced_accuracy': float(np.std(scores, ddof=1)),
        'fold_scores': scores,
        'feature_count': int(features.shape[1]),
    }
    print(f'{feature_name} | mean={np.mean(scores):.4f} | std={np.std(scores, ddof=1):.4f}')

ranking = sorted(results.items(), key=lambda item: item[1]['mean_balanced_accuracy'], reverse=True)
summary = {
    'experiment': 'prompt-normalized response-pattern features',
    'split_method': '5-fold GroupKFold by prompt_id',
    'feature_context': 'same-prompt group statistics; labels not used',
    'transductive_assumption': 'all responses for a prompt group are available at inference',
    'feature_names': FEATURE_NAMES,
    'results': results,
    'ranking': [{'configuration': key, **value} for key, value in ranking],
    'total_elapsed_seconds': round(time.time() - started, 2),
}
PATTERN_OUTPUT.write_text(json.dumps(summary, indent=2))
print('\\nFinal ranking:')
for position, (key, value) in enumerate(ranking, start=1):
    print(f'{position}. {key}: {value["mean_balanced_accuracy"]:.4f} +/- {value["std_balanced_accuracy"]:.4f}')
print(f'Saved results to {PATTERN_OUTPUT}')


## Interpretation

Prompt-relative features are useful only if the competition inference format exposes the complete response group. The grouped CV result will be compared with the 33.89% TF-IDF OOF baseline before any ensemble is attempted.
